# Logistic Regression training on unscaled features

Open the notebook in VS Code, select the project's `.venv` kernel, and run the four code cells in order with **Shift+Enter**. The model uses the existing 103 CTG features exactly as stored in the shared training CSV.

## 1. Set up
Import the libraries and locate the shared training CSV. Check the printed path before continuing.

In [9]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

relative_path = Path("data/processed/train.csv")
train_path = next(
    (folder / relative_path for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / relative_path).is_file()),
    None,
)
if train_path is None:
    raise FileNotFoundError("Open the repository folder in VS Code before running this notebook.")
print(f"Training data: {train_path}")

Training data: c:\Users\LENOVO\Desktop\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\data\processed\train.csv


## 2. Load and check the training data
Select the 103 feature columns by prefix and use `figo_label` as the target. Verify that the values are numeric and finite and that both labels are present. These checks leave the feature values unchanged.

In [10]:
train = pd.read_csv(train_path)
features = [column for column in train if column.startswith(("F_", "U_", "FU_"))]
if len(features) != 103:
    raise ValueError(f"Expected 103 CTG predictors, found {len(features)}.")

X_train = train[features]
y_train = train["figo_label"]
if y_train.isna().any() or set(y_train) != {0, 1}:
    raise ValueError("Expected both FIGO labels, 0 (normal) and 1 (abnormal).")
if not all(pd.api.types.is_numeric_dtype(X_train[column]) for column in features):
    raise ValueError("All CTG predictors must be numeric.")
if not np.isfinite(X_train.to_numpy(dtype=float)).all():
    raise ValueError("The training features contain missing or infinite values.")

print(f"Training shape: {X_train.shape}")
print(y_train.value_counts().sort_index().to_string())

Training shape: (14540, 103)
figo_label
0    6719
1    7821


## 3. Fit Logistic Regression
Fit the classifier directly on the training features. The solver uses L-BFGS, `C=1.0`, tolerance `1e-4`, and seed 42. If the first fit warns, the cell retries with a higher iteration limit. Check the printed iteration count and convergence status.

In [11]:
model_settings = {"solver": "lbfgs", "C": 1.0, "tol": 1e-4, "random_state": 42}
for max_iter in (1000, 5000):
    model = LogisticRegression(max_iter=max_iter, **model_settings)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X_train, y_train)

    convergence_warnings = [
        item for item in caught if issubclass(item.category, ConvergenceWarning)
    ]
    print(f"max_iter={max_iter}, iterations used={model.n_iter_[0]}")
    for item in caught:
        print(f"{item.category.__name__}: {item.message}")
    if not convergence_warnings:
        print("Training finished without a convergence warning.")
        break
else:
    print("The solver still warned at the final iteration limit.")

max_iter=1000, iterations used=143
Training finished without a convergence warning.


## 4. Evaluate on validation data
Predict labels for the shared validation recordings. Abnormal-class F1 is the primary comparison metric. Precision describes the correctness of abnormal predictions, sensitivity measures how many abnormal recordings are detected, and specificity measures how many normal recordings are identified correctly. Accuracy summarizes all predictions. The confusion matrix shows actual classes in rows and predicted classes in columns.

In [12]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

if "model" not in globals() or "convergence_warnings" not in globals():
    raise RuntimeError("Run the first three code cells in order to fit Logistic Regression.")
if convergence_warnings:
    raise RuntimeError("Resolve the training convergence warning before evaluating.")

val = pd.read_csv(train_path.with_name("val.csv"))
y_val = val["figo_label"]
predictions = model.predict(val[features])
metrics = pd.Series({
    "Abnormal F1 (primary)": f1_score(y_val, predictions, pos_label=1, zero_division=0),
    "Accuracy": accuracy_score(y_val, predictions),
    "Abnormal precision": precision_score(y_val, predictions, pos_label=1, zero_division=0),
    "Sensitivity (abnormal recall)": recall_score(y_val, predictions, pos_label=1, zero_division=0),
    "Specificity (normal recall)": recall_score(y_val, predictions, pos_label=0, zero_division=0),
}, name="Validation score")
print(metrics.to_string(float_format="{:.4f}".format))
print("\nConfusion matrix")
print(pd.DataFrame(
    confusion_matrix(y_val, predictions, labels=[0, 1]),
    index=["Actual normal", "Actual abnormal"],
    columns=["Predicted normal", "Predicted abnormal"],
))

Abnormal F1 (primary)           0.7711
Accuracy                        0.7618
Abnormal precision              0.7982
Sensitivity (abnormal recall)   0.7458
Specificity (normal recall)     0.7804

Confusion matrix
                 Predicted normal  Predicted abnormal
Actual normal                1123                 316
Actual abnormal               426                1250
